# Optimización de Modelos de Regresión
## Facultad de Ingeniería - Universidad Nacional de Asunción
### Inteligencia Artificial 2026

**Diego Stalder** | **Christian Torres**

---

## 📌 Introducción

Una vez que hemos preprocesado los datos y establecido un modelo base, el siguiente paso es mejorar su rendimiento. Esto implica seleccionar los mejores hiperparámetros y validar que nuestro modelo generalice correctamente a datos no vistos.

> **"La validación cruzada es el estándar de oro para evaluar el rendimiento de un modelo."**

### En este notebook abordaremos:

* ✓ **Validación Cruzada (Cross-Validation)**: Técnica para evaluar modelos de forma robusta
* ✓ **Grid Search**: Búsqueda exhaustiva de hiperparámetros
* ✓ **Random Search**: Búsqueda aleatoria más eficiente
* ✓ **Optimización con Optuna**: Búsqueda inteligente y automatizada

Utilizaremos el dataset **California Housing** para predecir el valor medio de las casas.

## 1. Configuración Inicial

In [3]:
# Importación de librerías necesarias
import pandas as pd
import numpy as np

# Modelos y métricas
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Para búsqueda de hiperparámetros
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Para optimización avanzada
import optuna
from optuna.samplers import TPESampler

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


h:\Anaconda\envs\deepf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga y Preparación de Datos

In [4]:
# Cargamos el dataset California Housing
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = pd.Series(california.target, name='MedHouseVal')

print("📊 Dataset California Housing:")
print(X.head())
print("\n📊 Variable objetivo:")
print(y.head())
print(f"\n📏 Dimensiones: {X.shape}")

📊 Dataset California Housing:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252   41.0000    6.9841     1.0238    322.0000    2.5556   37.8800   
1  8.3014   21.0000    6.2381     0.9719   2401.0000    2.1098   37.8600   
2  7.2574   52.0000    8.2881     1.0734    496.0000    2.8023   37.8500   
3  5.6431   52.0000    5.8174     1.0731    558.0000    2.5479   37.8500   
4  3.8462   52.0000    6.2819     1.0811    565.0000    2.1815   37.8500   

   Longitude  
0  -122.2300  
1  -122.2200  
2  -122.2400  
3  -122.2500  
4  -122.2500  

📊 Variable objetivo:
0   4.5260
1   3.5850
2   3.5210
3   3.4130
4   3.4220
Name: MedHouseVal, dtype: float64

📏 Dimensiones: (20640, 8)


In [5]:
# Dividimos en train y test (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Train set: {X_train.shape}")
print(f"📊 Test set: {X_test.shape}")

📊 Train set: (16512, 8)
📊 Test set: (4128, 8)


## 3. Modelo Base sin Optimización

Primero, establecemos un modelo base con parámetros por defecto para tener una referencia de rendimiento.

In [6]:
# Modelo base: Random Forest con parámetros por defecto
rf_base = RandomForestRegressor(random_state=42)
rf_base.fit(X_train, y_train)

# Predicciones
y_pred_base = rf_base.predict(X_test)

# Métricas
mse_base = mean_squared_error(y_test, y_pred_base)
rmse_base = np.sqrt(mse_base)
mae_base = mean_absolute_error(y_test, y_pred_base)
r2_base = r2_score(y_test, y_pred_base)

print("📊 Métricas del modelo base:")
print(f"   MSE:  {mse_base:.4f}")
print(f"   RMSE: {rmse_base:.4f}")
print(f"   MAE:  {mae_base:.4f}")
print(f"   R²:   {r2_base:.4f}")

📊 Métricas del modelo base:
   MSE:  0.2554
   RMSE: 0.5053
   MAE:  0.3275
   R²:   0.8051


## 4. Validación Cruzada (Cross-Validation)

La validación cruzada nos permite evaluar el modelo de manera más robusta, utilizando múltiples particiones de entrenamiento/validación.

In [7]:
# Configuramos la validación cruzada
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Evaluamos con cross-validation
cv_scores = cross_val_score(rf_base, X_train, y_train, cv=kfold, 
                            scoring='r2', n_jobs=-1)

print("📊 Resultados de validación cruzada (R²):")
for i, score in enumerate(cv_scores, 1):
    print(f"   Fold {i}: {score:.4f}")
print(f"\n   Media: {cv_scores.mean():.4f}")
print(f"   Desv. estándar: {cv_scores.std():.4f}")

📊 Resultados de validación cruzada (R²):
   Fold 1: 0.8002
   Fold 2: 0.8127
   Fold 3: 0.8089
   Fold 4: 0.7955
   Fold 5: 0.8054

   Media: 0.8045
   Desv. estándar: 0.0061


In [8]:
# También podemos obtener múltiples métricas
from sklearn.model_selection import cross_validate

scoring = {'r2': 'r2', 'neg_mse': 'neg_mean_squared_error', 
           'neg_mae': 'neg_mean_absolute_error'}

cv_results = cross_validate(rf_base, X_train, y_train, cv=kfold, 
                            scoring=scoring, n_jobs=-1)

print("📊 Resultados detallados de CV:")
print(f"   R²  - Media: {cv_results['test_r2'].mean():.4f} (±{cv_results['test_r2'].std():.4f})")
print(f"   MSE - Media: {-cv_results['test_neg_mse'].mean():.4f} (±{cv_results['test_neg_mse'].std():.4f})")
print(f"   MAE - Media: {-cv_results['test_neg_mae'].mean():.4f} (±{cv_results['test_neg_mae'].std():.4f})")

📊 Resultados detallados de CV:
   R²  - Media: 0.8045 (±0.0061)
   MSE - Media: 0.2613 (±0.0118)
   MAE - Media: 0.3351 (±0.0052)


## 5. Grid Search (Búsqueda Exhaustiva)

Grid Search evalúa todas las combinaciones posibles de hiperparámetros definidos en una cuadrícula.

In [9]:
# Definimos el modelo y el espacio de búsqueda
rf = RandomForestRegressor(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("🔍 Espacio de búsqueda Grid Search:")
print(f"   Total de combinaciones: {np.prod([len(v) for v in param_grid.values()])}")

🔍 Espacio de búsqueda Grid Search:
   Total de combinaciones: 108


In [10]:
# Configuramos Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# Entrenamos
grid_search.fit(X_train, y_train)

print("\n✅ Grid Search completado")

Fitting 5 folds for each of 108 candidates, totalling 540 fits

✅ Grid Search completado


In [11]:
# Mejores parámetros y rendimiento
print("📊 Resultados de Grid Search:")
print(f"   Mejores parámetros: {grid_search.best_params_}")
print(f"   Mejor R² (CV): {grid_search.best_score_:.4f}")

# Evaluamos en test
y_pred_grid = grid_search.predict(X_test)
r2_grid = r2_score(y_test, y_pred_grid)
rmse_grid = np.sqrt(mean_squared_error(y_test, y_pred_grid))

print(f"\n   Rendimiento en test:")
print(f"   R²:   {r2_grid:.4f} (vs base: {r2_base:.4f})")
print(f"   RMSE: {rmse_grid:.4f} (vs base: {rmse_base:.4f})")

📊 Resultados de Grid Search:
   Mejores parámetros: {'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
   Mejor R² (CV): 0.8053

   Rendimiento en test:
   R²:   0.8063 (vs base: 0.8051)
   RMSE: 0.5039 (vs base: 0.5053)


In [12]:
# DataFrame con resultados
results_grid = pd.DataFrame(grid_search.cv_results_)
print("📊 Top 5 combinaciones:")
print(results_grid[['params', 'mean_test_score', 'std_test_score']]
      .sort_values('mean_test_score', ascending=False)
      .head(5))

📊 Top 5 combinaciones:
                                               params  mean_test_score  \
92  {'max_depth': 30, 'min_samples_leaf': 2, 'min_...           0.8053   
11  {'max_depth': None, 'min_samples_leaf': 2, 'mi...           0.8052   
65  {'max_depth': 20, 'min_samples_leaf': 2, 'min_...           0.8052   
14  {'max_depth': None, 'min_samples_leaf': 2, 'mi...           0.8050   
56  {'max_depth': 20, 'min_samples_leaf': 1, 'min_...           0.8050   

    std_test_score  
92          0.0046  
11          0.0046  
65          0.0046  
14          0.0046  
56          0.0053  


## 6. Random Search (Búsqueda Aleatoria)

Random Search muestrea aleatoriamente combinaciones de hiperparámetros, siendo más eficiente que Grid Search para espacios de búsqueda grandes.

In [13]:
# Espacio de búsqueda más amplio para Random Search
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(50, 300, 10)],
    'max_depth': [None] + [int(x) for x in np.linspace(5, 50, 10)],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 6, 8],
    'max_features': ['sqrt', 'log2', None]
}

print(f"🔍 Espacio de búsqueda: {np.prod([len(v) for v in param_dist.values()])} combinaciones posibles")

🔍 Espacio de búsqueda: 8250 combinaciones posibles


In [14]:
# Configuramos Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,  # Número de combinaciones a probar
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)
print("\n✅ Random Search completado")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

✅ Random Search completado


In [15]:
print("📊 Resultados de Random Search:")
print(f"   Mejores parámetros: {random_search.best_params_}")
print(f"   Mejor R² (CV): {random_search.best_score_:.4f}")

# Evaluamos en test
y_pred_random = random_search.predict(X_test)
r2_random = r2_score(y_test, y_pred_random)
rmse_random = np.sqrt(mean_squared_error(y_test, y_pred_random))

print(f"\n   Rendimiento en test:")
print(f"   R²:   {r2_random:.4f}")
print(f"   RMSE: {rmse_random:.4f}")

📊 Resultados de Random Search:
   Mejores parámetros: {'n_estimators': 188, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 40}
   Mejor R² (CV): 0.8159

   Rendimiento en test:
   R²:   0.8148
   RMSE: 0.4927


In [16]:
# Comparación de tiempos
print("📊 Comparación de tiempos de búsqueda:")
print(f"   Grid Search:   {grid_search.refit_time_:.2f} segundos")
print(f"   Random Search: {random_search.refit_time_:.2f} segundos")

📊 Comparación de tiempos de búsqueda:
   Grid Search:   21.44 segundos
   Random Search: 11.37 segundos


## 7. Optimización con Optuna

Optuna utiliza métodos avanzados como **TPE (Tree-structured Parzen Estimator)** para optimizar hiperparámetros de manera inteligente, aprendiendo de pruebas anteriores.

In [17]:
# Definimos la función objetivo para Optuna
def objective(trial):
    # Definimos los hiperparámetros a optimizar
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    
    # Creamos y evaluamos el modelo
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    
    # Validación cruzada
    scores = cross_val_score(model, X_train, y_train, 
                             cv=5, scoring='r2', n_jobs=-1)
    
    return scores.mean()

print("🔍 Configurando estudio de Optuna...")

🔍 Configurando estudio de Optuna...


In [18]:
# Creamos el estudio y optimizamos
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n✅ Optimización con Optuna completada")

[I 2026-03-14 20:21:47,365] A new study created in memory with name: no-name-e3022c19-7c69-4fd0-8409-e485abb0f980
Best trial: 0. Best value: 0.796911:   2%|▏         | 1/50 [00:02<02:20,  2.86s/it]

[I 2026-03-14 20:21:50,229] Trial 0 finished with value: 0.7969114970713042 and parameters: {'n_estimators': 144, 'max_depth': 48, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7969114970713042.


Best trial: 1. Best value: 0.806657:   4%|▍         | 2/50 [00:08<03:36,  4.51s/it]

[I 2026-03-14 20:21:55,890] Trial 1 finished with value: 0.8066569637098251 and parameters: {'n_estimators': 267, 'max_depth': 32, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8066569637098251.


Best trial: 1. Best value: 0.806657:   6%|▌         | 3/50 [00:14<04:01,  5.13s/it]

...

[I 2026-03-14 20:25:42,795] Trial 47 finished with value: 0.8120625417578526 and parameters: {'n_estimators': 209, 'max_depth': 45, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 44 with value: 0.8155569314244607.


Best trial: 44. Best value: 0.815557:  98%|█████████▊| 49/50 [04:02<00:06,  6.59s/it]

[I 2026-03-14 20:25:49,655] Trial 48 finished with value: 0.8107555358993723 and parameters: {'n_estimators': 262, 'max_depth': 49, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 44 with value: 0.8155569314244607.


Best trial: 44. Best value: 0.815557: 100%|██████████| 50/50 [04:07<00:00,  4.95s/it]

[I 2026-03-14 20:25:54,745] Trial 49 finished with value: 0.8040253113293815 and parameters: {'n_estimators': 221, 'max_depth': 43, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'log2'}. Best is trial 44 with value: 0.8155569314244607.

✅ Optimización con Optuna completada


In [19]:
print("📊 Resultados de Optuna:")
print(f"   Mejor valor (R² CV): {study.best_value:.4f}")
print(f"   Mejores parámetros:")
for key, value in study.best_params.items():
    print(f"      {key}: {value}")

📊 Resultados de Optuna:
   Mejor valor (R² CV): 0.8156
   Mejores parámetros:
      n_estimators: 187
      max_depth: 46
      min_samples_split: 2
      min_samples_leaf: 1
      max_features: sqrt


In [20]:
# Entrenamos el mejor modelo encontrado por Optuna
best_rf_optuna = RandomForestRegressor(**study.best_params, random_state=42)
best_rf_optuna.fit(X_train, y_train)

# Evaluamos en test
y_pred_optuna = best_rf_optuna.predict(X_test)
r2_optuna = r2_score(y_test, y_pred_optuna)
rmse_optuna = np.sqrt(mean_squared_error(y_test, y_pred_optuna))

print("📊 Rendimiento del modelo optimizado con Optuna:")
print(f"   R² en test:   {r2_optuna:.4f}")
print(f"   RMSE en test: {rmse_optuna:.4f}")

📊 Rendimiento del modelo optimizado con Optuna:
   R² en test:   0.8141
   RMSE en test: 0.4936


## 8. Comparación de Métodos

In [21]:
# Creamos tabla comparativa
comparacion = pd.DataFrame({
    'Método': ['Base', 'Grid Search', 'Random Search', 'Optuna'],
    'R² (test)': [r2_base, r2_grid, r2_random, r2_optuna],
    'RMSE (test)': [rmse_base, rmse_grid, rmse_random, rmse_optuna],
    'Mejor R² (CV)': [cv_scores.mean(), grid_search.best_score_, 
                      random_search.best_score_, study.best_value]
})

print("📊 Comparación de métodos de optimización:")
print(comparacion.to_string(index=False))

📊 Comparación de métodos de optimización:
       Método  R² (test)  RMSE (test)  Mejor R² (CV)
         Base     0.8051       0.5053         0.8045
  Grid Search     0.8063       0.5039         0.8053
Random Search     0.8148       0.4927         0.8159
       Optuna     0.8141       0.4936         0.8156


In [22]:
# Análisis de importancia de características del mejor modelo
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf_optuna.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 Importancia de características (mejor modelo):")
print(feature_importance.to_string(index=False))

📊 Importancia de características (mejor modelo):
   feature  importance
    MedInc      0.3608
  Latitude      0.1327
 Longitude      0.1296
  AveOccup      0.1225
  AveRooms      0.1180
  HouseAge      0.0564
 AveBedrms      0.0446
Population      0.0355


## 9. Optimización de Múltiples Modelos

Podemos usar Optuna para comparar y optimizar diferentes tipos de modelos automáticamente.

In [23]:
def objective_multi_model(trial):
    # Seleccionamos el tipo de modelo
    model_type = trial.suggest_categorical('model_type', 
                                           ['random_forest', 'gradient_boosting', 'ridge'])
    
    if model_type == 'random_forest':
        params = {
            'n_estimators': trial.suggest_int('rf_n_estimators', 50, 300),
            'max_depth': trial.suggest_int('rf_max_depth', 5, 50),
            'min_samples_split': trial.suggest_int('rf_min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('rf_min_samples_leaf', 1, 10)
        }
        model = RandomForestRegressor(**params, random_state=42)
        
    elif model_type == 'gradient_boosting':
        params = {
            'n_estimators': trial.suggest_int('gb_n_estimators', 50, 300),
            'max_depth': trial.suggest_int('gb_max_depth', 3, 15),
            'learning_rate': trial.suggest_float('gb_learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('gb_subsample', 0.5, 1.0)
        }
        model = GradientBoostingRegressor(**params, random_state=42)
        
    else:  # ridge
        params = {
            'alpha': trial.suggest_float('ridge_alpha', 0.1, 10.0, log=True)
        }
        model = Ridge(**params, random_state=42)
    
    # Evaluamos con cross-validation
    scores = cross_val_score(model, X_train, y_train, 
                             cv=5, scoring='r2', n_jobs=-1)
    
    return scores.mean()

print("🔍 Optimizando múltiples modelos con Optuna...")

🔍 Optimizando múltiples modelos con Optuna...


In [24]:
study_multi = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_multi.optimize(objective_multi_model, n_trials=50, show_progress_bar=True)

print("\n✅ Optimización multi-modelo completada")
print(f"\n📊 Mejor resultado:")
print(f"   Modelo: {study_multi.best_params['model_type']}")
print(f"   R² CV: {study_multi.best_value:.4f}")
print(f"   Parámetros: {study_multi.best_params}")

[I 2026-03-14 20:26:02,261] A new study created in memory with name: no-name-6763411e-282d-4b16-bf7e-60b0a2d5c665
Best trial: 0. Best value: 0.821404:   2%|▏         | 1/50 [00:06<05:21,  6.55s/it]

[I 2026-03-14 20:26:08,814] Trial 0 finished with value: 0.8214038456001369 and parameters: {'model_type': 'gradient_boosting', 'gb_n_estimators': 200, 'gb_max_depth': 5, 'gb_learning_rate': 0.055238410897498764, 'gb_subsample': 0.5290418060840998}. Best is trial 0 with value: 0.8214038456001369.


Best trial: 0. Best value: 0.821404:   6%|▌         | 3/50 [00:11<02:28,  3.16s/it]

[I 2026-03-14 20:26:13,887] Trial 1 finished with value: 0.7981739466679565 and parameters: {'model_type': 'random_forest', 'rf_n_estimators': 55, 'rf_max_depth': 49, 'rf_min_samples_split': 17, 'rf_min_samples_leaf': 3}. Best is trial 0 with value: 0.8214038456001369.
[I 2026-03-14 20:26:14,046] Trial 2 finished with value: 0.6114858747291253 and parameters: {'model_type': 'ridge', 'ridge_alpha': 1.1207606211860568}. Best is trial 0 with value: 0.8214038456001369.


Best trial: 0. Best value: 0.821404:  10%|█         | 5/50 [00:12<00:58,  1.31s/it]

...

[I 2026-03-14 20:36:30,937] Trial 47 finished with value: 0.8374416433270003 and parameters: {'model_type': 'gradient_boosting', 'gb_n_estimators': 276, 'gb_max_depth': 8, 'gb_learning_rate': 0.034584349036775756, 'gb_subsample': 0.7773922714091569}. Best is trial 40 with value: 0.8394928423272521.


Best trial: 48. Best value: 0.839508:  98%|█████████▊| 49/50 [10:48<00:17, 17.08s/it]

[I 2026-03-14 20:36:51,090] Trial 48 finished with value: 0.8395080263181356 and parameters: {'model_type': 'gradient_boosting', 'gb_n_estimators': 277, 'gb_max_depth': 8, 'gb_learning_rate': 0.040381160215622575, 'gb_subsample': 0.7655119835932316}. Best is trial 48 with value: 0.8395080263181356.


Best trial: 48. Best value: 0.839508: 100%|██████████| 50/50 [10:57<00:00, 13.15s/it]

[I 2026-03-14 20:36:59,902] Trial 49 finished with value: 0.7034219016346912 and parameters: {'model_type': 'random_forest', 'rf_n_estimators': 191, 'rf_max_depth': 6, 'rf_min_samples_split': 7, 'rf_min_samples_leaf': 7}. Best is trial 48 with value: 0.8395080263181356.

✅ Optimización multi-modelo completada

📊 Mejor resultado:
   Modelo: gradient_boosting
   R² CV: 0.8395
   Parámetros: {'model_type': 'gradient_boosting', 'gb_n_estimators': 277, 'gb_max_depth': 8, 'gb_learning_rate': 0.040381160215622575, 'gb_subsample': 0.7655119835932316}


## 10. Resumen y Conclusiones

### Comparativa de métodos:

| Método | Ventajas | Desventajas | Mejor uso |
|--------|----------|-------------|-----------|
| **Grid Search** | Exhaustivo, garantiza encontrar el óptimo en el espacio definido | Computacionalmente costoso, maldición de la dimensionalidad | Espacios pequeños (<100 combinaciones) |
| **Random Search** | Más eficiente que Grid, buena cobertura del espacio | No garantiza encontrar el óptimo global | Espacios medianos (100-1000 combinaciones) |
| **Optuna** | Inteligente, adaptativo, muy eficiente | Curva de aprendizaje, requiere configuración | Espacios grandes y complejos |

### Mejores prácticas:

1. **Siempre usar validación cruzada** para evaluar modelos
2. **Comenzar con Random Search** para explorar el espacio
3. **Refinar con Grid Search** alrededor de las mejores regiones
4. **Usar Optuna** para problemas complejos o cuando el tiempo es limitado
5. **Validar en test set** solo al final, después de toda la optimización

### Resultados obtenidos:

* El modelo base alcanzó un R² de **{:.4f}**
* Con optimización, mejoramos a **{:.4f}**
* La mejora relativa fue del **{:.1f}%**

La optimización de hiperparámetros es un paso crucial en cualquier pipeline de Machine Learning, y la elección del método depende del problema específico y los recursos computacionales disponibles.".format(r2_base, r2_optuna, (r2_optuna - r2_base)/r2_base*100)

In [25]:
# Guardamos el mejor modelo para uso futuro
import joblib

joblib.dump(best_rf_optuna, 'mejor_modelo_california.pkl')
print("✅ Modelo guardado como 'mejor_modelo_california.pkl'")

✅ Modelo guardado como 'mejor_modelo_california.pkl'
